In [ ]:
from typing import Annotated
from langgraph.graph import StateGraph, START
from langgraph.graph.message import add_messages
from dotenv import load_dotenv
from IPython.display import Image, display
import gradio as gr
from langgraph.prebuilt import ToolNode, tools_condition
import requests
import os
from langchain_openai import ChatOpenAI
from typing import TypedDict
from langchain_core.tools import Tool
from langchain_community.utilities import GoogleSerperAPIWrapper
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

In [ ]:
load_dotenv(override=True)

In [ ]:
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_user = os.getenv("PUSHOVER_USER")
pushover_url = "https://api.pushover.net/1/messages.json"

def push(text: str):
    """Send a push notification to the user"""
    requests.post(pushover_url, data = {"token": pushover_token, "user": pushover_user, "message": text})

In [ ]:
#Pushover Tool
tool_push = Tool(
        name="send_push_notification",
        func=push,
        description="Send a push notification"
    )

In [ ]:
#Search Tool
serper = GoogleSerperAPIWrapper()
tool_search =Tool(
        name="search",
        func=serper.run,
        description="Search online"
    )

In [ ]:
tools = [tool_search, tool_push]

In [ ]:
db_path = "memory-test.db"
conn = sqlite3.connect(db_path, check_same_thread=False)
sql_memory = SqliteSaver(conn)

In [ ]:
# Step 1: Define the State object
class State(TypedDict):
    messages: Annotated[list, add_messages]

In [ ]:
# Step 2: Start the Graph Builder
graph_builder = StateGraph(State)

In [ ]:
llm = ChatOpenAI(model="gpt-oss-20b")
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# Step 3 : Create a node
def chatbot(state: State):
    print(state)
    return {"messages": [llm_with_tools.invoke(state["messages"])]}

graph_builder.add_node("chatbot", chatbot)
graph_builder.add_node("tools", ToolNode(tools=tools))

In [ ]:
# Step 4 : Create a edge
graph_builder.add_conditional_edges( "chatbot", tools_condition, "tools")
graph_builder.add_edge("tools", "chatbot")
graph_builder.add_edge(START, "chatbot")

In [ ]:
# Step 5 : Compile the Graph
graph = graph_builder.compile(checkpointer=sql_memory)

In [ ]:
Image(graph.get_graph().draw_mermaid_png())

In [ ]:
config = {"configurable": {"thread_id": "2"}}

def chat(user_input: str, history):
    result = graph.invoke({"messages": [{"role": "user", "content": user_input}]}, config=config)
    return result["messages"][-1].content


gr.ChatInterface(chat).launch()